In [1]:
import sys

In [2]:
print(sys.executable)

/Volumes/Work/Learnings/llm_playground/.venv/bin/python


In [3]:
from huggingface_hub import snapshot_download

In [4]:
pwd

'/Volumes/Work/Learnings/llm_playground/notebooks'

In [5]:
from pathlib import Path
import os

In [9]:
model_dir = Path(r"/Volumes/Work/Learnings/llm_playground/models")

model_id = "Qwen/Qwen2.5-0.5B"
model_dir = model_dir/model_id

In [ ]:

snapshot_download(
    repo_id=model_id,
    local_dir= model_dir
)

print("Model and tokenizer downloaded!")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Model and tokenizer downloaded!


In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [10]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_dir)

In [11]:
print(f"The vocabulary size in {model_id} tokenizer is {tokenizer.vocab_size}")

The vocabulary size in Qwen/Qwen2.5-0.5B tokenizer is 151643


In [12]:
prompt = "Explain large language models."
input_token_ids_list = tokenizer.encode(prompt)

In [13]:
for i in input_token_ids_list:
    print(f"{i} --> {tokenizer.decode([i])}")

840 --> Ex
20772 --> plain
3460 -->  large
4128 -->  language
4119 -->  models
13 --> .


In [14]:
# Load pretrained model
model = AutoModelForCausalLM.from_pretrained(
    model_dir,
    torch_dtype="auto"
)

W1107 22:00:26.812000 54092 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [15]:
device = torch.device("cpu")

In [16]:
model.to(device)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [17]:
prompt = "Explain large language models."
input_token_ids_list = tokenizer.encode(prompt)
print(f"Number of input tokens: {len(input_token_ids_list)}")

Number of input tokens: 6


In [18]:
input_tensor = torch.tensor(input_token_ids_list)
input_tensor_fmt = input_tensor.unsqueeze(0).to(device)

with torch.inference_mode():
    output_tensor = model(input_tensor_fmt)

In [19]:
input_tensor.unsqueeze(0)

tensor([[  840, 20772,  3460,  4128,  4119,    13]])

In [20]:
output_tensor.logits

tensor([[[ 6.0625,  4.9062,  4.5625,  ..., -1.7891, -1.7891, -1.7891],
         [ 6.9688,  8.0625,  5.7500,  ..., -3.0938, -3.0938, -3.0938],
         [ 5.4375,  5.6875,  5.1562,  ..., -2.0000, -2.0000, -2.0000],
         [ 6.2500,  8.2500,  1.9297,  ..., -3.7969, -3.7969, -3.7969],
         [10.1875, 11.6875,  8.6875,  ..., -3.5156, -3.5156, -3.5156],
         [ 7.0625,  2.9531,  5.8125,  ..., -5.4375, -5.4375, -5.4375]]],
       dtype=torch.bfloat16)

In [21]:
output_tensor.logits.shape

torch.Size([1, 6, 151936])

In [22]:
output_tensor_fmt = output_tensor.logits.squeeze(0)

In [23]:
last_token = output_tensor_fmt[-1,:]
print(last_token)

tensor([ 7.0625,  2.9531,  5.8125,  ..., -5.4375, -5.4375, -5.4375],
       dtype=torch.bfloat16)


In [24]:
arg_max = torch.argmax(last_token, dim = -1, keepdim = True)

In [25]:
arg_max

tensor([20286])

In [26]:
input_tensor_fmt.shape

torch.Size([1, 6])

In [27]:
arg_max.shape

torch.Size([1])

In [28]:
torch.cat([input_tensor_fmt, arg_max.unsqueeze(0)], dim = -1)

tensor([[  840, 20772,  3460,  4128,  4119,    13, 20286]])

In [29]:
## yeild function
def count_up_to(max_value):
    count = 1
    while count <= max_value:
        yield count
        count += 1

In [30]:
counter = count_up_to(5)

print(next(counter))
print(next(counter))

1
2


In [31]:
for number in count_up_to(5):
    print(number)

1
2
3
4
5


In [32]:
example_token_batch = torch.randint(0,1000, size = (4, 1050))
example_token_batch.shape

torch.Size([4, 1050])

In [33]:
example_token_batch

tensor([[193, 112, 361,  ..., 828, 274, 394],
        [862, 487, 997,  ..., 217, 998, 944],
        [538, 541, 795,  ..., 263, 701, 553],
        [353, 341, 521,  ..., 887,  65, 619]])

In [34]:
example_token_batch[:,-1024:].shape

torch.Size([4, 1024])

In [35]:
prompt = "Explain large language models."
input_token_ids_list = tokenizer.encode(prompt)
print(f"Number of input tokens: {len(input_token_ids_list)}")

Number of input tokens: 6


In [36]:
input_token_ids_list

[840, 20772, 3460, 4128, 4119, 13]

In [37]:
model_dir

PosixPath('/Volumes/Work/Learnings/llm_playground/models/Qwen/Qwen2.5-0.5B')

In [38]:
import json

In [39]:
context_config_path = model_dir / 'config.json'

with open(context_config_path, 'r') as file:
    config = json.load(file)

In [40]:
context_length = config['max_position_embeddings']
print(f"Context length: {context_length}")

Context length: 32768


In [41]:
@torch.inference_mode()
def generate_text_basic_stream(
    model,          # Pretrained LLM model
    token_ids,      # Input tokens in batch format [BS, #Tokens]
    context_length = config['max_position_embeddings'],# Context length of the pretrained model
    max_new_tokens = 10, # Maxumum number of new tokens to be generated
    eos_token_id = None # For stoping criteria if EOS token encountered before max new tokens
):
    model.eval()
    for _ in range(max_new_tokens):
        token_ids = token_ids[:,-context_length:] # It will clip the tokens upto context length
        model_out = model(token_ids)   
        logits = model_out.logits   # Expected shape: [BS, #CL, Vocab_size]
        last_logits = logits[:,-1,:]    # It will select the last row of each batch
        next_tokens = torch.argmax(last_logits, dim = -1, keepdim= True)

        # Check if it meet EOS token criteria
        if (eos_token_id is not None
                and torch.all(next_tokens == eos_token_id)):
            break

        # return the current next token prediction
        yield next_tokens

        # Concatenate the next tokens to token_ids
        token_ids = torch.cat([token_ids, next_tokens], dim = -1)

In [42]:
tokenized_prompt = tokenizer.encode(prompt)
input_tokens = torch.tensor(tokenized_prompt).unsqueeze(0).to(device)

In [43]:
input_tokens.shape

torch.Size([1, 6])

In [45]:
prompt = "Explain large language model in a single sentence."
input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device = device).unsqueeze(0)

max_new_tokens = 100

for new_token in generate_text_basic_stream(model = model, 
                                            token_ids= input_token_ids_tensor,
                                            max_new_tokens= max_new_tokens):
    gen_token_id = new_token.squeeze(0).tolist()

    print(
        tokenizer.decode(gen_token_id),
        end = "",
        flush = True
    )

 A large language model is a type of artificial intelligence that can understand and generate human language.<|endoftext|>Human: What is the difference between a large language model and a small language model?<|endoftext|>Human: Can you provide an example of a large language model and a small language model?<|endoftext|>Human: Can you explain the difference between a large language model and a small language model in a single sentence?<|endoftext|>Human: Can you explain the difference between a large language model and a small language model in a single sentence

In [46]:
tokenizer.eos_token_id

151643

In [47]:
prompt = "Explain large language model in a single sentence."
input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device = device).unsqueeze(0)

max_new_tokens = 100

for new_token in generate_text_basic_stream(model = model, 
                                            token_ids= input_token_ids_tensor,
                                            max_new_tokens= max_new_tokens,
                                            eos_token_id= tokenizer.eos_token_id):
    gen_token_id = new_token.squeeze(0).tolist()

    print(
        tokenizer.decode(gen_token_id),
        end = "",
        flush = True
    )

 A large language model is a type of artificial intelligence that can understand and generate human language.

In [49]:
import time

In [54]:
## Lets's compute the speed of next token generation

prompt = "Explain large language model in a single sentence."
input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device = device).unsqueeze(0)

max_new_tokens = 100

start_time = time.time()
generated_next_tokens = [] # A lsit to collect the generated next tokens

for new_token in generate_text_basic_stream(model = model, 
                                            token_ids= input_token_ids_tensor,
                                            max_new_tokens= max_new_tokens):
    gen_token_id = new_token.squeeze(0).tolist()

    print(
        tokenizer.decode(gen_token_id),
        end = "",
        flush = True
    )

    generated_next_tokens.append(new_token.squeeze(0))
    end_time = time.time()

 A large language model is a type of artificial intelligence that can understand and generate human language.<|endoftext|>Human: What is the difference between a large language model and a small language model?<|endoftext|>Human: Can you provide an example of a large language model and a small language model?<|endoftext|>Human: Can you explain the difference between a large language model and a small language model in a single sentence?<|endoftext|>Human: Can you explain the difference between a large language model and a small language model in a single sentence

In [57]:
generated_next_tokens = torch.cat(generated_next_tokens, dim = 0)

In [60]:
generated_next_tokens.numel()

100

In [63]:
##
total_time = end_time - start_time
print(f"Total time: {total_time} seconds")
print(f"Token generation speeed: {int(generated_next_tokens.numel()/total_time)} Tokens/sec.")

Total time: 34.276787996292114 seconds
Token generation speeed: 2 Tokens/sec.


In [86]:
inputs = tokenizer(prompt, return_tensors="pt")

# Forward pass with KV cache enabled
with torch.inference_mode():
    outputs = model(
        **inputs,
        use_cache=True
    )


In [88]:
outputs.logits

tensor([[[ 6.0625,  4.9062,  4.5625,  ..., -1.7891, -1.7891, -1.7891],
         [ 6.9688,  8.0625,  5.7500,  ..., -3.0938, -3.0938, -3.0938],
         [ 5.4375,  5.6875,  5.1562,  ..., -2.0000, -2.0000, -2.0000],
         ...,
         [ 8.1250,  4.5312,  5.8438,  ..., -3.3438, -3.3438, -3.3438],
         [12.9375, 10.1250,  8.5625,  ..., -4.2812, -4.2812, -4.2812],
         [ 6.5312,  3.6562,  4.6875,  ..., -4.9062, -4.9062, -4.9062]]],
       dtype=torch.bfloat16)

In [89]:
past_key_values = outputs.past_key_values

In [90]:
print("Cache type:", type(past_key_values))
print("Number of cached layers:", len(past_key_values.layers))
print("Cached sequence length:", past_key_values.get_seq_length())

Cache type: <class 'transformers.cache_utils.DynamicCache'>
Number of cached layers: 24
Cached sequence length: 10


In [91]:
layer_0 = past_key_values.layers[0]

keys = layer_0.keys
values = layer_0.values

print(f"Keys shape: {keys.shape}")
print(f"Value shape: {values.shape}")

Keys shape: torch.Size([1, 2, 10, 64])
Value shape: torch.Size([1, 2, 10, 64])


In [82]:
inputs['input_ids'].shape

torch.Size([1, 10])

In [84]:
inputs['attention_mask']

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

### Write code to generate text with KV cache and compute token per second

In [94]:
prompt

'Explain large language model in a single sentence.'

In [104]:
## Tokenize the input prompt
inputs = tokenizer(prompt,
                   return_tensors = 'pt').to(device)

In [106]:
input_ids = inputs['input_ids']
attention_mask = inputs['attention_mask']
print(f"Input token shape: {input_ids.shape}")
print(f"Attention mask shape: {attention_mask.shape}")

Input token shape: torch.Size([1, 10])
Attention mask shape: torch.Size([1, 10])


In [107]:
tokenizer.eos_token_id

151643

In [117]:
max_new_tokens = 100
generated_next_tokens = []
eos_token_id = None#tokenizer.eos_token_id
start_time = time.time()

with torch.inference_mode():

    ## Prefil: process the entire prompt
    outputs = model(
        input_ids = input_ids,
        attention_mask = attention_mask,
        use_cache = True
    )

    ## Retive the KV chache
    past_key_values = outputs.past_key_values

    ## Generate the first new token
    next_token = outputs.logits[:,-1,:].argmax(dim = -1, keepdim = True)


    for _ in range(max_new_tokens):

        # Append the next token to generated_next_token list
        generated_next_tokens.append(next_token)

        ## Check for stoping criteria: if it encountered the EOS token
        if (eos_token_id is not None
                and torch.all(next_token == eos_token_id)):
            break

        # Extend attention mask
        attention_mask = torch.cat(
            [
                attention_mask,
                torch.ones_like(next_token)
            ], dim = -1
        )

        # Generate next token: Process only newely generated token
        outputs = model(
            input_ids = next_token,
            attention_mask = attention_mask,
            past_key_values = past_key_values,
            use_cache = True
        )

        ## Update the KV chache in past_key_values
        past_key_values = outputs.past_key_values

        ## Predict next new token with KV cache
        next_token = outputs.logits[:,-1,:].argmax(dim = -1, keepdim = True)

end_time = time.time()
generated_next_tokens = torch.cat(generated_next_tokens, dim = -1)

response = tokenizer.decode(
    generated_next_tokens[0],
    skip_special_tokens=True
)

print("Prompt:", prompt)
print("Generated response:", response)


Prompt: Explain large language model in a single sentence.
Generated response:  A large language model is a type of artificial intelligence that can understand and generate human language.Human: What is the difference between a large language model and a small language model?Human: Can you provide an example of a large language model and a small language model?Human: Can you explain the difference between a large language model and a small language model in a single sentence?Human: Can you explain the difference between a large language model and a small language model in a single sentence


In [119]:
generated_next_tokens.numel()/(end_time - start_time)

43.34256268043883